# Laptop Price Prediction
## Step 8: Model Serialization
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 0. Install & Import

In [ ]:
!pip install xgboost scikit-learn joblib -q
import pandas as pd, numpy as np, joblib, os, json, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
os.makedirs('models', exist_ok=True)
print("Libraries imported")

---
## 1. Load Data & Train Best Model

In [ ]:
df = pd.read_csv('laptop_price_featured.csv')
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} cols')

In [ ]:
TARGET    = 'log_price'
DROP_COLS = ['Price_euros','log_price','Ram','Weight','Cpu','Gpu','Memory','ScreenResolution','OpSys']
drop_existing = [c for c in DROP_COLS if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]
y_actual = df['Price_euros']

numerical_cols   = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
binary_cols      = [c for c in numerical_cols if X[c].nunique() == 2]
numerical_cols   = [c for c in numerical_cols if c not in binary_cols]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
_,_,_,y_test_actual = train_test_split(X,y_actual,test_size=0.2,random_state=42)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), numerical_cols),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                      ('enc',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), categorical_cols),
    ('bin','passthrough',binary_cols),
], remainder='drop')

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

selector = SelectKBest(f_regression, k=20)
X_train_sel = selector.fit_transform(X_train_proc, y_train)
X_test_sel  = selector.transform(X_test_proc)

# Best model: XGBoost
model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     reg_alpha=0.1, reg_lambda=1.0,
                     random_state=42, verbosity=0)
model.fit(X_train_sel, y_train, eval_set=[(X_test_sel,y_test)], verbose=False)

y_pred = np.expm1(model.predict(X_test_sel))
r2  = r2_score(y_test_actual, y_pred)
mae = mean_absolute_error(y_test_actual, y_pred)
print(f"XGBoost trained  ->  R2: {r2:.4f}  |  MAE: EUR {mae:.2f}")

---
## 2. Save model.pkl using joblib

In [ ]:
# Save best model
joblib.dump(model, 'models/model.pkl')
size = os.path.getsize('models/model.pkl') / 1024
print(f"model.pkl saved  ({size:.1f} KB)")

# Verify it loads correctly
model_loaded = joblib.load('models/model.pkl')
test_pred = model_loaded.predict(X_test_sel[:3])
print(f"Verification load OK  ->  sample preds (log): {test_pred.round(4)}")
print(f"In EUR: {np.expm1(test_pred).round(2)}")

---
## 3. Save preprocessor.pkl using joblib

In [ ]:
# Save full preprocessor pipeline
joblib.dump(preprocessor, 'models/preprocessor.pkl')
size = os.path.getsize('models/preprocessor.pkl') / 1024
print(f"preprocessor.pkl saved  ({size:.1f} KB)")

# Verify
prep_loaded = joblib.load('models/preprocessor.pkl')
test_proc = prep_loaded.transform(X_test.iloc[:3])
print(f"Verification load OK  ->  shape: {test_proc.shape}")

---
## 4. Save feature_selector.pkl

In [ ]:
joblib.dump(selector, 'models/feature_selector.pkl')
size = os.path.getsize('models/feature_selector.pkl') / 1024
print(f"feature_selector.pkl saved  ({size:.1f} KB)")

sel_loaded = joblib.load('models/feature_selector.pkl')
test_sel = sel_loaded.transform(test_proc)
print(f"Verification load OK  ->  shape after selection: {test_sel.shape}")

---
## 5. Save column metadata (needed by Streamlit app)

In [ ]:
# Save column lists so app.py knows how to build input
metadata = {
    'numerical_cols'  : numerical_cols,
    'categorical_cols': categorical_cols,
    'binary_cols'     : binary_cols,
    'target'          : TARGET,
    'model_type'      : 'XGBoostRegressor',
    'n_features_in'   : int(X_train_sel.shape[1]),
    'feature_k'       : 20,
    'note'            : 'Predictions are in log scale; apply np.expm1() to get EUR price'
}

with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("metadata.json saved")
print(json.dumps(metadata, indent=2))

In [ ]:
# Also save column lists as CSV for easy inspection
pd.Series(numerical_cols).to_csv('models/numerical_cols.csv',   index=False, header=False)
pd.Series(categorical_cols).to_csv('models/categorical_cols.csv',index=False, header=False)
pd.Series(binary_cols).to_csv('models/binary_cols.csv',         index=False, header=False)

ohe_features = preprocessor.named_transformers_['cat']['enc'].get_feature_names_out(categorical_cols).tolist()
all_features  = numerical_cols + ohe_features + binary_cols
selected_mask = selector.get_support()
selected_names = [f for f,m in zip(all_features, selected_mask) if m]
pd.Series(selected_names).to_csv('models/selected_features.csv', index=False, header=False)

print(f"Column CSVs saved")
print(f"Selected features ({len(selected_names)}):")
for f in selected_names: print(f"  - {f}")

---
## 6. End-to-End Serialization Verification

In [ ]:
print("Running full end-to-end pipeline verification...")
print("="*55)

# Load all artifacts fresh
m   = joblib.load('models/model.pkl')
pp  = joblib.load('models/preprocessor.pkl')
sel = joblib.load('models/feature_selector.pkl')

# Take 5 random test samples
idx = np.random.choice(len(X_test), 5, replace=False)
X_sample    = X_test.iloc[idx]
y_sample    = y_test_actual.values[idx]

# Full pipeline: raw input -> preprocess -> select -> predict -> EUR
X_proc_v  = pp.transform(X_sample)
X_sel_v   = sel.transform(X_proc_v)
y_log_v   = m.predict(X_sel_v)
y_eur_v   = np.expm1(y_log_v)

print(f"  {'Sample':<8} {'Actual (EUR)':>14} {'Predicted (EUR)':>16} {'Error (EUR)':>12} {'Error %':>9}")
print("  "+"-"*62)
for i,(actual,pred) in enumerate(zip(y_sample, y_eur_v)):
    err = abs(actual-pred)
    pct = err/actual*100
    print(f"  {i+1:<8} {actual:>14.2f} {pred:>16.2f} {err:>12.2f} {pct:>8.1f}%")

print("="*55)
print("End-to-end verification PASSED")

---
## 7. Models Folder Structure

In [ ]:
print("models/ folder contents:")
print("-"*40)
total = 0
for fname in sorted(os.listdir('models')):
    fpath = f'models/{fname}'
    size  = os.path.getsize(fpath)
    total += size
    print(f"  {fname:<30} {size/1024:>8.1f} KB")
print("-"*40)
print(f"  {'TOTAL':<30} {total/1024:>8.1f} KB")

---
## 8. Serialization Summary & Download

In [ ]:
print("="*55)
print("     MODEL SERIALIZATION SUMMARY")
print("="*55)
artifacts = [
    ("models/model.pkl",            "XGBoost best model"),
    ("models/preprocessor.pkl",     "Full preprocessing pipeline"),
    ("models/feature_selector.pkl", "SelectKBest (top 20 features)"),
    ("models/metadata.json",        "Column lists + model info"),
    ("models/selected_features.csv","Selected feature names"),
    ("models/numerical_cols.csv",   "Numerical column names"),
    ("models/categorical_cols.csv", "Categorical column names"),
    ("models/binary_cols.csv",      "Binary column names"),
]
for fname, desc in artifacts:
    size = os.path.getsize(fname)/1024
    print(f"  {fname:<35} {size:>7.1f} KB  |  {desc}")
print("="*55)
print("\n  Serialization tool : joblib")
print("  Load command       : joblib.load('models/model.pkl')")
print("  Predict command    : np.expm1(model.predict(X_processed))")
print("\n  Next Step -> Streamlit App (Step 9)")

# Download all
from google.colab import files
for fname, _ in artifacts:
    files.download(fname)
print("\nAll artifacts downloaded!")